In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
import scipy.linalg as sci
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pickle

In [3]:
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [5]:
def compute_mse(input_values, quantized_values):
    """MSE 계산 함수"""
    return torch.mean((input_values - quantized_values) ** 2)

def update_centroid(mask, distribution, input_values):
    pdf_val = torch.exp(distribution.log_prob(input_values[mask]))
    denom = torch.sum(input_values[mask] * pdf_val)
    
    return denom/torch.sum(pdf_val)

normal_dist = torch.distributions.Normal(0, 1)
input_values = torch.linspace(-15,15, steps=10000)  # 10,000개의 샘플 생성

# 다양한 양자화 레벨에서 실험
quantization_levels = [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]       
centroid_per_level={}
boundary_per_level={}
tol = torch.tensor(1e-6)
for levels in quantization_levels:
    centroids = torch.linspace(-3, 3, steps = levels)
    mse_prev = torch.tensor(0)
    mse_curr = torch.tensor(0)
    while True:
        quantized_values = torch.zeros_like(input_values)
        boundaries = (centroids[:-1] + centroids[1:]) / 2
        
        for i in range(len(centroids)):
            if i == 0:
                mask = torch.where(input_values <= boundaries[0].item())[0]
            elif i == len(centroids) - 1:
                mask = torch.where(input_values > boundaries[-1].item())[0]
            else:
                mask = torch.where((input_values > boundaries[i-1].item()) & (input_values <= boundaries[i].item()))[0]
            
            new_centroid = update_centroid(mask, normal_dist, input_values)
            centroids[i] = new_centroid
                
            quantized_values[mask] = centroids[i]
        
        # MSE 계산
        mse_prev = mse_curr
        mse_curr = compute_mse(input_values, quantized_values)
        
        if torch.abs(mse_curr - mse_prev) / mse_curr < tol:
            break
    centroid_per_level[levels] = centroids
    boundary_per_level[levels] = (centroids[:-1] + centroids[1:]) / 2
    
print(centroid_per_level)
### centroid_per_level : 2^B level Gaussian Lloyd-Max quantizer level을 가지는 dictionary

{2: tensor([-0.7979,  0.7979]), 4: tensor([-1.5124, -0.4537,  0.4537,  1.5124]), 8: tensor([-2.1635, -1.3560, -0.7647, -0.2481,  0.2481,  0.7647,  1.3560,  2.1635]), 16: tensor([-2.7811, -2.1227, -1.6720, -1.3074, -0.9867, -0.6913, -0.4099, -0.1357,
         0.1357,  0.4099,  0.6913,  0.9867,  1.3074,  1.6720,  2.1227,  2.7811]), 32: tensor([-3.5073, -2.9746, -2.6308, -2.3643, -2.1386, -1.9366, -1.7455, -1.5587,
        -1.3748, -1.1907, -1.0052, -0.8213, -0.6388, -0.4578, -0.2768, -0.0927,
         0.0927,  0.2768,  0.4578,  0.6388,  0.8213,  1.0052,  1.1907,  1.3748,
         1.5587,  1.7455,  1.9366,  2.1386,  2.3643,  2.6308,  2.9746,  3.5073]), 64: tensor([-3.8944, -3.4119, -3.1065, -2.8811, -2.7017, -2.5522, -2.4227, -2.3080,
        -2.2065, -2.1122, -2.0192, -1.9263, -1.8334, -1.7404, -1.6475, -1.5545,
        -1.4616, -1.3687, -1.2757, -1.1828, -1.0898, -0.9969, -0.9039, -0.8095,
        -0.7135, -0.6176, -0.5232, -0.4287, -0.3328, -0.2368, -0.1424, -0.0480,
         0.0480,  

In [6]:
with open('Lloyd-Max quantizer levels', 'wb') as level:
    pickle.dump(centroid_per_level, level)
with open('Lloyd-Max quantizer boundaries', 'wb') as bound:
    pickle.dump(boundary_per_level, bound)

In [7]:
Mainlobe_UE= np.array([0,0],dtype=np.float32) #Center of the AoD range for K users
HalfBW_UE = np.array([30.0,30.0],dtype=np.float32) #Half of the AoD range for K users
low=Mainlobe_UE[0]-HalfBW_UE[0]
high=Mainlobe_UE[0]+HalfBW_UE[0]
quantization_levels = [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]     


U_quantizer_level = {}
U_quantizer_boundary = {}
for q_levels in quantization_levels:
    boundary = torch.linspace(low, high, q_levels+1)
    U_quantizer_boundary[q_levels] = boundary
    U_quantizer_level[q_levels] = (boundary[1:] + boundary[:-1])/2


In [8]:
with open('Uniform quantizer levels', 'wb') as fw:
    pickle.dump(U_quantizer_level, fw)
with open('Uniform quantizer boundaries', 'wb') as fw:
    pickle.dump(U_quantizer_boundary, fw)